In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import nltk
import glob

In [3]:
import sys
sys.path.append('lib/')

In [4]:
from detection.relationshipextraction import RelationshipDiscovery, GateExtractor, CleanDicts, rmToRelationCSV
from detection.schema import Term, Concept, df_to_concepts, cleaningPlaceStr, conceptsToGazetteer
from detection.worldumls import umlsConceptCleanner, isEnglish, worldConceptCleanner
from detection.worldumls import ClearnWorldKGGazetteer
#import detection.observationclustering

In [16]:
path_to_relation_folder = "../data/neo4j/"
path_to_covid_journalobservationcsv = "../data/neo4j/covid_observations_journal.csv"
path_to_covid_medicalobservationcsv = "../data/neo4j/covid_observations_medical.csv"
path_to_covid_socialobservationcsv = "../data/neo4j/covid_observations_social.csv"
path_to_monkeypox_journalobservationcsv = "../data/neo4j/monkeypox_observations_journal.csv"
path_to_monkeypox_medicalobservationcsv = "../data/neo4j/monkeypox_observations_medical.csv"
path_to_monkeypox_socialobservationcsv = "../data/neo4j/monkeypox_observations_social.csv"
path_to_covid_llama3_jounralobservationcsv = "../data/neo4j/covid_llama3_observations_journal.csv"

In [6]:
from lib.kgce.schema.semantic.neo4jclasses import Neo4jRelation
from lib.kgce.neo4j.handler import Neo4jWrapper

In [7]:
from neo4j import GraphDatabase
from tqdm import tqdm


class Neo4jWrapper:

    def __init__(self, uri, userName, password):
        self.uri = uri
        self.userName = userName
        self.password = password
        # Connect to the neo4j database server
        self.graphDB_Driver  = GraphDatabase.driver(uri, auth=(userName, password)) 
        
    def sendQuery(self, cql_commands):
        result = []
        done_queries = []
        with self.graphDB_Driver.session() as graphDB_Session:
            for cqlCreate in tqdm(cql_commands):
                try:
                    result += [graphDB_Session.run(cqlCreate).to_df()]
                    done_queries.append(cqlCreate)
                except Exception as e:
                    tqdm.write(str(e))
                    tqdm.write(cqlCreate)
                    result += [str(e)]
        return result
    
    def closeConnection(self):
        self.graphDB_Driver.close()

In [8]:
neowrapper = Neo4jWrapper(uri="bolt://localhost:7687",userName="neo4j",password="test")

In [9]:
def GetObservationFromSource(neowrapper,source, filterValue):
    strQuery = """MATCH (n:Country)<-[r:hasPresence]-(c) 
        WHERE toInteger(r.intensity) >= {0} AND r.source = "{1}"
        RETURN n.wkgs_nameEn as System_Name, n.id, c.name, c.id, r.intensity as intensity;""".format(
        filterValue, source)
    result = neowrapper.sendQuery([strQuery])
    df_result_journal = result[0].groupby(['System_Name','n.id'],as_index=False).agg(list)
    return df_result_journal

In [10]:
df_observation_journal_covid_llama3 = GetObservationFromSource(neowrapper,"Journal_COVID_LLAMA3",8)

100%|█████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.28s/it]


In [17]:
df_observation_journal_covid_llama3.to_csv(path_to_covid_llama3_jounralobservationcsv)

In [18]:
df_observation_social_monkey = GetObservationFromSource(neowrapper,"Social_Monkeypox",10)
df_observation_social_monkey.to_csv(path_to_monkeypox_socialobservationcsv)

100%|█████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.71it/s]
